# PDFormer on PEMS-BAY (Kaggle) - comparison baseline for HGP

Runs the **authors' PDFormer model code (BUAABIGSCity/PDFormer, MIT) unmodified** on PEMS-BAY, with the same chronological 70/10/20 split and the same common training settings as HGP, and reports test MAE / RMSE / MAPE at 15, 30 and 60 minutes.

Changes are limited to environment/data plumbing (listed in section 2); model, loss, attention, masks and metrics are untouched.

**Two stages (set `STAGE`)**
- `caches` - CPU is enough. Builds the one-off caches the authors' code needs for a new dataset: the DTW matrix (parallelised, identical values) and the KShape pattern keys (single-threaded, expected to take several hours). Saves them under `/kaggle/working/pdformer_caches/`. Save the notebook version, then add that output as an input of the training run.
- `train` - GPU. Reuses the caches and trains/tests. If no pattern keys are found they are built first.

**Before running**: Internet On (clones the PDFormer repository); add a dataset holding `pems-bay.h5` and `adj_mx_bay.pkl`; for `train` choose a GPU (T4 x1 / P100).

In [ ]:
STAGE = "train"            # "caches" (CPU) or "train" (GPU)
DATASET = "PEMSBAY"
SEED = 42
MAX_EPOCH = 40             # HGP protocol; PDFormer's own default is 200
REPO_URL = "https://github.com/BUAABIGSCity/PDFormer.git"


## 1. Environment

In [ ]:
import glob, json, os, shutil, subprocess, sys
import torch
print(sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "", "| cpus", os.cpu_count())
if STAGE == "train":
    assert torch.cuda.is_available(), "turn the GPU accelerator on"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastdtw", "tslearn", "tabulate", "tensorboard", "tqdm"], check=True)


## 2. Code and data
Edits to the cloned repository: `np.float` -> `float` (removed in numpy >= 1.24) and the Ray hyper-parameter-search imports made optional (used only by the tuning entry point). New files: `prepare_pemsbay.py` (data conversion), `precompute_dtw.py` (parallel DTW), `precompute_keys.py`, and `pdformer_lowmem.py` / `run_lowmem.py` - the same data windows, split, padding, scaler and clustering as the original loader, but cut out of one matrix per batch instead of stored as a 30 GB array (verified identical on a slice by `tests/check_lowmem.py`).

In [ ]:
WORK = "/kaggle/working"
CODE = "/tmp/pdformer"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE)

p = "libcity/data/dataset/traffic_state_datatset.py"
text = open(p).read()        # read first: open(p, "w") would empty the file before it is read
open(p, "w").write(text.replace("dtype=np.float)", "dtype=float)"))

try:
    subprocess.run([sys.executable, "-c", "from ray.tune.suggest.hyperopt import HyperOptSearch"], check=True, capture_output=True)
except Exception:
    for p in ("libcity/pipeline/pipeline.py", "libcity/executor/pdformer_executor.py", "libcity/executor/traffic_state_executor.py"):
        lines = open(p).read().split("\n")
        lines = [("try:\n    " + l + "\nexcept Exception:\n    pass") if l.startswith(("from ray", "import ray")) else l for l in lines]
        open(p, "w").write("\n".join(lines))
    print("ray tuning imports made optional")

files = [f for f in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(f)]
TRAFFIC = [f for f in files if f.lower().endswith(".h5") and "pems" in os.path.basename(f).lower() and "meta" not in os.path.basename(f).lower()][0]
ADJ = [f for f in files if f.lower().endswith(".pkl") and "adj" in os.path.basename(f).lower()][0]
print(TRAFFIC, ADJ)


In [ ]:
%%writefile prepare_pemsbay.py
"""Convert PEMS-BAY (DCRNN release) into the LibCity atomic files PDFormer reads. Data preparation only; no model code is touched.

    python prepare_pemsbay.py --traffic pems-bay.h5 --adjacency adj_mx_bay.pkl --name PEMSBAY

Writes raw_data/<name>/{<name>.geo, <name>.rel, <name>.dyna, config.json}.
Values are the raw speeds (mph) with missing readings left as 0, which PDFormer's masked loss and metrics treat as missing.
The road links are the non-zero entries of the DCRNN adjacency (the same road graph HGP uses); bidir=true in the model config
symmetrises them.
"""
import argparse
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument("--traffic", required=True)
ap.add_argument("--adjacency", required=True)
ap.add_argument("--name", default="PEMSBAY")
ap.add_argument("--sensors", type=int, default=None, help="keep the first K sensors (smoke test only)")
ap.add_argument("--steps", type=int, default=None, help="keep the first K time steps (smoke test only)")
a = ap.parse_args()

table = pd.read_hdf(a.traffic)
with open(a.adjacency, "rb") as f:
    content = pickle.load(f, encoding="latin1")
adj = np.asarray(content[2], dtype=np.float32)
if a.sensors:
    table, adj = table.iloc[:, :a.sensors], adj[:a.sensors, :a.sensors]
if a.steps:
    table = table.iloc[:a.steps]
T, N = table.shape
speed = np.nan_to_num(table.values.astype(np.float32), nan=0.0)

out = Path("raw_data") / a.name
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"geo_id": range(N), "type": "Point", "coordinates": "[]"}).to_csv(out / f"{a.name}.geo", index=False)
src, dst = np.nonzero(adj)
pd.DataFrame({"rel_id": range(len(src)), "type": "geo", "origin_id": src, "destination_id": dst,
              "cost": adj[src, dst]}).to_csv(out / f"{a.name}.rel", index=False)
times = np.array([t.strftime("%Y-%m-%dT%H:%M:%SZ") for t in table.index])
with open(out / f"{a.name}.dyna", "w", newline="") as f:
    f.write("dyna_id,type,time,entity_id,traffic_speed\n")
    k = 0
    for j in range(N):
        chunk = pd.DataFrame({"dyna_id": np.arange(k, k + T), "type": "state", "time": times, "entity_id": j,
                              "traffic_speed": speed[:, j]})
        chunk.to_csv(f, header=False, index=False)
        k += T
json.dump({"geo": {"including_types": ["Point"], "Point": {}},
           "rel": {"including_types": ["geo"], "geo": {"cost": "num"}},
           "dyna": {"including_types": ["state"], "state": {"entity_id": "geo_id", "traffic_speed": "num"}},
           "info": {"data_col": ["traffic_speed"], "weight_col": "cost", "data_files": [a.name], "geo_file": a.name,
                    "rel_file": a.name, "output_dim": 1, "time_intervals": 300, "init_weight_inf_or_zero": "zero",
                    "set_weight_link_or_dist": "link", "calculate_weight_adj": False, "weight_adj_epsilon": 0}},
          open(out / "config.json", "w"), indent=2)
print(f"{out}: {T} steps x {N} sensors, {len(src)} road links, {(speed == 0).mean():.4%} missing")


In [ ]:
%%writefile precompute_dtw.py
"""Parallel version of PDFormerDataset._get_dtw: same fastdtw call on the same daily-average curves, so the matrix is identical,
but the rows are spread over all CPU cores. Run once before run_model.py; run_model.py then loads the cache instead of recomputing.

    python precompute_dtw.py --name PEMSBAY [--out path.npy]
"""
import argparse
import os
from multiprocessing import Pool

import numpy as np
import pandas as pd
from fastdtw import fastdtw

DATA = None


def load(name):
    df = pd.read_csv(f"./raw_data/{name}/{name}.dyna")[["time", "entity_id", "traffic_speed"]]
    n = df["entity_id"].nunique()
    values = df["traffic_speed"].values.astype(float).reshape(n, -1)
    return values.T[:, :, None]


def init(data_mean):
    global DATA
    DATA = data_mean


def row(i):
    n = DATA.shape[1]
    return i, [fastdtw(DATA[:, i, :], DATA[:, j, :], radius=6)[0] for j in range(i, n)]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--out", default=None)
    ap.add_argument("--workers", type=int, default=os.cpu_count())
    a = ap.parse_args()
    out = a.out or f"./libcity/cache/dataset_cache/dtw_{a.name}.npy"
    df = load(a.name)
    day = 24 * (3600 // 300)
    data_mean = np.mean([df[day * i: day * (i + 1)] for i in range(df.shape[0] // day)], axis=0)
    n = data_mean.shape[1]
    dtw = np.zeros((n, n))
    with Pool(a.workers, initializer=init, initargs=(data_mean,)) as pool:
        for k, (i, vals) in enumerate(pool.imap_unordered(row, range(n)), 1):
            dtw[i, i:] = vals
            if k % 25 == 0 or k == n:
                print(f"{k}/{n} rows", flush=True)
    for i in range(n):
        for j in range(i):
            dtw[i][j] = dtw[j][i]
    os.makedirs(os.path.dirname(out), exist_ok=True)
    np.save(out, dtw)
    print("saved", out)


In [ ]:
%%writefile precompute_keys.py
"""Build the one-off dataset caches (DTW matrix, windowed arrays, KShape pattern keys) through PDFormer's own data pipeline,
without training. Needs no GPU, so it can run in a CPU-only session; run_model.py then finds the caches and starts training at once.

    python precompute_keys.py --dataset PEMSBAY --config_file PEMSBAY_hgp
"""
import argparse

from libcity.config import ConfigParser
import libcity.data.dataset as datasets
from libcity.data import get_dataset

from pdformer_lowmem import PDFormerLowMemDataset

datasets.PDFormerDataset = PDFormerLowMemDataset

ap = argparse.ArgumentParser()
ap.add_argument("--dataset", required=True)
ap.add_argument("--config_file", required=True)
a = ap.parse_args()
config = ConfigParser("traffic_state_pred", "PDFormer", a.dataset, a.config_file, True, True, {"gpu": False})
dataset = get_dataset(config)
dataset.get_data()
print("caches written to libcity/cache/dataset_cache/")


In [ ]:
%%writefile pdformer_lowmem.py
"""Memory-light data loading for PDFormer. The model, loss, masks and metrics are untouched.

PDFormerDataset materialises every (input, target) window as one dense float64 array. For PEMS-BAY that is
52,093 windows x 12 steps x 325 sensors x 9 features x 8 bytes = 14.6 GB for the inputs and the same for the targets
(plus copies made while splitting and padding), which does not fit in a 30 GB machine. This subclass keeps the single
(time x sensor x feature) matrix and cuts each window out of it when a batch asks for it.

It reproduces the original pipeline exactly:
  * the same feature matrix (speed, time of day, one-hot day of week) built by the original loader methods;
  * the same chronological split (round() of the sample counts) and the same repeat-the-last-sample padding of every split;
  * the same StandardScaler, fitted on the speed channel of all training input windows (computed from per-row window
    counts instead of from the materialised windows - the same mean and standard deviation);
  * the same pattern-key clustering call on the same candidate windows.
tests/check_lowmem.py compares it with the original on a small slice.
"""
import os

import numpy as np
from torch.utils.data import DataLoader

from libcity.data.batch import Batch
from libcity.data.dataset.pdformer_dataset import PDFormerDataset
from libcity.data.list_dataset import ListDataset


class Windows:
    """Lazy list of (x, y) windows over a (T, N, F) matrix; window w covers rows w..w+in-1 (x) and w+in..w+in+out-1 (y)."""

    def __init__(self, matrix, first, count, n_in, n_out, batch_size):
        self.matrix, self.first, self.count = matrix, first, count
        self.n_in, self.n_out = n_in, n_out
        self.length = count + (batch_size - count % batch_size) % batch_size

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        if i < 0:
            i += self.length
        w = self.first + min(i, self.count - 1)
        x = self.matrix[w: w + self.n_in]
        y = self.matrix[w + self.n_in: w + self.n_in + self.n_out]
        return x.copy(), y.copy()


class PDFormerLowMemDataset(PDFormerDataset):

    def _feature_matrix(self):
        if isinstance(self.data_files, list):
            data_files = self.data_files.copy()
        else:
            data_files = [self.data_files].copy()
        mats = []
        for filename in data_files:
            df = self._load_dyna(filename)
            if self.load_external:
                df = self._add_external_information(df, None)
            mats.append(df)
        return np.concatenate(mats, axis=0)

    def _split_counts(self, num_samples):
        test_rate = 1 - self.train_rate - self.eval_rate
        num_test = round(num_samples * test_rate)
        num_train = round(num_samples * self.train_rate)
        num_val = num_samples - num_test - num_train
        return num_train, num_val, num_test

    def get_data(self):
        n_in, n_out = self.input_window, self.output_window
        F = self._feature_matrix()
        num_samples = F.shape[0] - n_in - n_out + 1
        num_train, num_val, num_test = self._split_counts(num_samples)
        first_train = int(num_train * (1 - self.part_train_rate))
        n_train = num_train - first_train
        self._logger.info("Dataset created (lazy windows)")
        self._logger.info("windows: train {}, eval {}, test {}, feature dim {}".format(
            n_train, num_val, num_test, F.shape[-1]))

        self.feature_dim = F.shape[-1]
        self.ext_dim = self.feature_dim - self.output_dim

        # StandardScaler of the original: mean/std of the speed channel over all training input windows
        d = self.output_dim
        rows = np.arange(first_train, num_train + n_in - 1)
        counts = np.minimum(rows, num_train - 1) - np.maximum(first_train, rows - (n_in - 1)) + 1
        speed = F[rows[0]: rows[-1] + 1, :, :d].astype(np.float64)
        total = counts.sum() * speed.shape[1] * d
        mean = (speed.sum(axis=(1, 2)) * counts).sum() / total
        var = ((speed ** 2).sum(axis=(1, 2)) * counts).sum() / total - mean ** 2
        self.scaler = self._get_scalar_from(mean, np.sqrt(var))
        self.ext_scaler = self._get_scalar(self.ext_scaler_type, F[:1, :, d:], F[:1, :, d:])
        del speed

        F[..., :d] = self.scaler.transform(F[..., :d])

        bs = self.batch_size
        train = Windows(F, first_train, n_train, n_in, n_out, bs)
        val = Windows(F, num_train, num_val, n_in, n_out, bs)
        test = Windows(F, num_samples - num_test, num_test, n_in, n_out, bs)

        def collator(items):
            batch = Batch(self.feature_name, None, None)
            for item in items:
                batch.append(item)
            batch.padding()
            return batch

        self.train_dataloader = DataLoader(ListDataset(train), batch_size=bs, num_workers=self.num_workers,
                                           collate_fn=collator, shuffle=True)
        self.eval_dataloader = DataLoader(ListDataset(val), batch_size=bs, num_workers=self.num_workers,
                                          collate_fn=collator, shuffle=True)
        self.test_dataloader = DataLoader(ListDataset(test), batch_size=bs, num_workers=self.num_workers,
                                          collate_fn=collator, shuffle=False)
        self.num_batches = len(self.train_dataloader)

        self.pattern_key_file = os.path.join(
            './libcity/cache/dataset_cache/', 'pattern_keys_{}_{}_{}_{}_{}_{}'.format(
                self.cluster_method, self.dataset, self.cand_key_days, self.s_attn_size, self.n_cluster,
                self.cluster_max_iter))
        if not os.path.exists(self.pattern_key_file + '.npy'):
            from tslearn.clustering import KShape, TimeSeriesKMeans
            k = min(self.cand_key_days * self.points_per_day, n_train)
            idx = np.arange(first_train, first_train + k)[:, None] + np.arange(self.s_attn_size)[None, :]
            cand = F[idx][..., :self.output_dim]                       # (k, s_attn_size, N, output_dim)
            pattern_cand_keys = cand.swapaxes(1, 2).reshape(-1, self.s_attn_size, self.output_dim)
            self._logger.info("Clustering...")
            if self.cluster_method == "kshape":
                km = KShape(n_clusters=self.n_cluster, max_iter=self.cluster_max_iter).fit(pattern_cand_keys)
            else:
                km = TimeSeriesKMeans(n_clusters=self.n_cluster, metric="softdtw",
                                      max_iter=self.cluster_max_iter).fit(pattern_cand_keys)
            self.pattern_keys = km.cluster_centers_
            np.save(self.pattern_key_file, self.pattern_keys)
            self._logger.info("Saved at file " + self.pattern_key_file + ".npy")
        else:
            self.pattern_keys = np.load(self.pattern_key_file + ".npy")
            self._logger.info("Loaded file " + self.pattern_key_file + ".npy")
        return self.train_dataloader, self.eval_dataloader, self.test_dataloader

    def _get_scalar_from(self, mean, std):
        from libcity.utils.normalization import StandardScaler as Std
        assert self.scaler_type == "standard", "the low-memory loader reproduces the standard scaler only"
        self._logger.info('StandardScaler mean: ' + str(mean) + ', std: ' + str(std))
        return Std(mean=mean, std=std)


In [ ]:
%%writefile run_lowmem.py
"""run_model.py with the memory-light dataset class registered (no library file is edited). Same command-line arguments."""
import runpy

import libcity.data.dataset as datasets

from pdformer_lowmem import PDFormerLowMemDataset

datasets.PDFormerDataset = PDFormerLowMemDataset
runpy.run_path("run_model.py", run_name="__main__")


## 3. Config: HGP common settings, everything else from the PDFormer paper / code

In [ ]:
cfg = {
    "dataset_class": "PDFormerDataset",
    "input_window": 12,
    "output_window": 12,
    "train_rate": 0.7,
    "eval_rate": 0.1,
    "learner": "adamw",
    "learning_rate": 0.0003,
    "weight_decay": 0.0001,
    "batch_size": 16,
    "max_epoch": 40,
    "lr_warmup_epoch": 1,
    "use_early_stop": True,
    "patience": 6,
    "clip_grad_norm": True,
    "max_grad_norm": 5,
    "seed": 42,
    "embed_dim": 64,
    "add_time_in_day": True,
    "add_day_in_week": True,
    "step_size": 2279,
    "bidir": True,
    "far_mask_delta": 7,
    "geo_num_heads": 4,
    "sem_num_heads": 2,
    "t_num_heads": 2,
    "cluster_method": "kshape",
    "cand_key_days": 14,
    "type_ln": "pre",
    "set_loss": "huber",
    "huber_delta": 2,
    "mode": "single",
    "drop": 0.1
}
cfg['max_epoch'] = MAX_EPOCH
cfg['seed'] = SEED
json.dump(cfg, open(f'{DATASET}_hgp.json', 'w'), indent=4)
print(json.dumps(cfg))

## 4. Data and one-off caches

In [ ]:
def run(cmd):
    with subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as pr:
        for line in pr.stdout:
            print(line, end="")
    assert pr.returncode == 0, f"{cmd[1]} exited with {pr.returncode}"

CACHE = "libcity/cache/dataset_cache"
os.makedirs(CACHE, exist_ok=True)
for f in glob.glob(f"/kaggle/input/**/*{DATASET}*.npy", recursive=True):
    shutil.copy(f, CACHE); print("restored", os.path.basename(f))

run([sys.executable, "prepare_pemsbay.py", "--traffic", TRAFFIC, "--adjacency", ADJ, "--name", DATASET])
if not os.path.exists(f"{CACHE}/dtw_{DATASET}.npy"):
    run([sys.executable, "precompute_dtw.py", "--name", DATASET])
    os.makedirs(f"{WORK}/pdformer_caches", exist_ok=True)   # keep the 70-minute matrix even if a later step fails
    shutil.copy(f"{CACHE}/dtw_{DATASET}.npy", f"{WORK}/pdformer_caches/"); print("saved dtw matrix")
if STAGE == "caches" and not glob.glob(f"{CACHE}/pattern_keys_*_{DATASET}_*.npy"):
    run([sys.executable, "precompute_keys.py", "--dataset", DATASET, "--config_file", f"{DATASET}_hgp"])
if STAGE == "caches":
    os.makedirs(f"{WORK}/pdformer_caches", exist_ok=True)
    for f in glob.glob(f"{CACHE}/*{DATASET}*.npy"):
        shutil.copy(f, f"{WORK}/pdformer_caches/"); print("saved", os.path.basename(f))
    print("caches done - save this notebook version and add its output as an input of the training run")

## 5. Train and test (STAGE = train)
If the pattern keys are not cached, the clustering runs first. Each epoch is 2,279 batches; the log prints the seconds per epoch - check the first epoch against the 12-hour session limit (40 epochs maximum, early stopping with patience 6).

In [ ]:
if STAGE == "train":
    run([sys.executable, "run_lowmem.py", "--task", "traffic_state_pred", "--model", "PDFormer", "--dataset", DATASET,
         "--config_file", f"{DATASET}_hgp", "--evaluator", "TrafficStateEvaluator"])

## 6. Results (test split; steps 3 / 6 / 12 = 15 / 30 / 60 min)

In [ ]:
import numpy as np
if STAGE == "train":
    npz = sorted(glob.glob(f"{CODE}/libcity/cache/*/evaluate_cache/*_predictions.npz"), key=os.path.getmtime)[-1]
    z = np.load(npz); pred, truth = z["prediction"][..., 0], z["truth"][..., 0]
    print(npz, pred.shape)
    valid = truth > 0
    rows = {}
    for h in (3, 6, 12):
        p, t, m = pred[:, h-1], truth[:, h-1], valid[:, h-1]
        e = (p - t)[m]
        rows[h] = dict(mae=float(np.abs(e).mean()), rmse=float(np.sqrt((e**2).mean())),
                       mape=float((np.abs(e) / t[m])[t[m] >= 1].mean() * 100))
        print(f"h={h:2d} ({5*h:2d} min)  MAE {rows[h]['mae']:.3f}  RMSE {rows[h]['rmse']:.3f}  MAPE {rows[h]['mape']:.2f}%")
    # same summary as HGP (hgp/metrics.py): mean of the three per-horizon metrics, and RMSE growth h12 / h3
    rows["mean"] = {k: float(np.mean([rows[h][k] for h in (3, 6, 12)])) for k in ("mae", "rmse", "mape")}
    rows["error_growth_rmse"] = rows[12]["rmse"] / rows[3]["rmse"]
    print("mean of h3/h6/h12:", rows["mean"], "| RMSE growth h12/h3:", round(rows["error_growth_rmse"], 3))
    e = (pred - truth)[valid]
    rows["all_12_steps"] = dict(mae=float(np.abs(e).mean()), rmse=float(np.sqrt((e**2).mean())))
    print("pooled over all 12 steps (PDFormer's own 'average' mode):", rows["all_12_steps"])
    out = f"{WORK}/pdformer_{DATASET}"
    os.makedirs(out, exist_ok=True)
    json.dump(rows, open(f"{out}/test_metrics.json", "w"), indent=2)
    shutil.copy(npz, f"{out}/test_predictions.npz")
    for f in glob.glob(f"{CODE}/libcity/cache/*/evaluate_cache/*.csv") + glob.glob(f"{CODE}/libcity/log/*.log"):
        shutil.copy(f, out)
    shutil.copy(f"{DATASET}_hgp.json", out)